In [4]:
import os
import mne
import numpy as np
import pandas as pd
from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from tqdm.notebook import tqdm

In [6]:
# create folder to store results if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ2"
if not os.path.exists(result_path):
    os.makedirs(result_path)

# create folder to store plots if not exist
result_plot= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ2/plot"
if not os.path.exists(result_plot):
    os.makedirs(result_plot)


In [7]:
df_skill = pd.read_csv(f"C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data_skill.csv")
df_filtered = df_skill[["Participant", "Algorithm", "SkillScore", "EEG", "CrossEEG"]]
df_skill = df_skill[["Participant", "SkillScore"]]
df_skill = df_skill.drop_duplicates()
# assign skill level
quantile = df_skill["SkillScore"].quantile([0.33, 0.66])
lower = quantile[0.33]
upper = quantile[0.66]

df_skill['SkillLevel'] = np.select([df_skill['SkillScore'] < lower, (df_skill['SkillScore'] >= lower) & (df_skill['SkillScore'] <= upper), df_skill['SkillScore'] > upper],
                                 ['Novice', 'Intermediate', 'Expert'],
                                 default='Intermediate')
df_skilled = df_skill.copy()
df_filtered = pd.merge(df_filtered, df_skilled[['Participant', 'SkillLevel']], on='Participant', how='left')
df_filtered = df_filtered[["Participant", "Algorithm", "SkillLevel", "EEG", "CrossEEG"]]
df_filtered

,Participant,Algorithm,SkillLevel,EEG,CrossEEG
0,1,IsPrime,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1,1,SiebDesEratosthenes,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
2,1,IsAnagram,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
3,1,RemoveDoubleChar,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
4,1,BinToDecimal,Intermediate,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
...,...,...,...,...,...
1067,71,DumpSorting,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1068,71,BinomialCoefficient,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1069,71,IsAnagram,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1070,71,ArrayAverage,Expert,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...


In [8]:
montage_path = result_path + '/../../Master-Thesis/AC-64.bvef'
montage = mne.channels.read_custom_montage(montage_path, head_size= 0.085)

In [9]:
def get_epochs(df_filtered):
    # Define epochs
    epochs = {}
    for subject in tqdm(df_filtered['Participant'].unique(), total=len(df_filtered['Participant'].unique())):
        print(f"Processing subject {subject}")
        raw_files = df_filtered[df_filtered['Participant'] == subject]['EEG']

        # Initialize a list to store epochs for each raw file
        subject_epochs = []
        freq_min = 4
        freq_max = 50 
        for raw_file in raw_files:
            raw = mne.io.read_raw_fif(raw_file, preload=True)
            montage = mne.channels.read_custom_montage(montage_path, head_size=0.085)
            # montage.plot()
            raw.set_montage(montage)

            # Apply bandpass filter to the raw data
            raw.filter(freq_min, freq_max, fir_design='firwin')

            # get raw channel data and do mean average referencing
            eeg_data_raw = raw.get_data()
            eeg_data_ref = eeg_data_raw - np.mean(eeg_data_raw, axis=0)

            # extract channel names of eeg_data
            channel_names = list(raw.to_data_frame().columns[1:])

            # create temporal eeg raw for cutting data into epochs
            tmp_raw = mne.io.RawArray(eeg_data_ref, raw.info, verbose='ERROR')

            # Convert annotations to events
            events = np.array([(0, 0, 1)])

            # Considered min. duration of the Participant's code comprehension as the time window
            time_window = 4                                                                                                                                                                                                                                                    

            # calculate EEG Signal duration
            eeg_duration = raw.n_times / raw.info['sfreq']

            # calculate midpoint of the signal
            midpoint = eeg_duration/2

            #Set time window's starting and end point to choose the middle part of the EEG signal
            tmin = midpoint - (time_window/2)
            tmax = midpoint + (time_window/2)
            
            raw_epochs = mne.Epochs(tmp_raw, events, event_id = 1, tmin=tmin, tmax=tmax, baseline=None, preload=True)
            # Append the epochs to the list
            subject_epochs.append(raw_epochs)

        # Store the epochs for this subject
        epochs[subject] = subject_epochs



In [20]:
df_raw = df_filtered[df_filtered["Participant"] == 1]


In [21]:
epochs = {}
for algorithm in tqdm(df_raw['Algorithm'], total=len(df_raw['Algorithm'])):
    print(f"Processing  {algorithm}")
    raw_files = df_raw[df_raw['Algorithm'] == algorithm]['EEG']

    # Initialize a list to store epochs for each raw file
    subject_epochs = []
    freq_min = 4
    freq_max = 50 
    for raw_file in raw_files:
        raw = mne.io.read_raw_fif(raw_file, preload=True)
        montage = mne.channels.read_custom_montage(montage_path, head_size=0.085)
        # montage.plot()
        raw.set_montage(montage)

        # Apply bandpass filter to the raw data
        raw.filter(freq_min, freq_max, fir_design='firwin')

        # get raw channel data and do mean average referencing
        eeg_data_raw = raw.get_data()
        eeg_data_ref = eeg_data_raw - np.mean(eeg_data_raw, axis=0)

        # extract channel names of eeg_data
        channel_names = list(raw.to_data_frame().columns[1:])

        # create temporal eeg raw for cutting data into epochs
        tmp_raw = mne.io.RawArray(eeg_data_ref, raw.info, verbose='ERROR')

        # Convert annotations to events
        events = np.array([(0, 0, 1)])

        # Considered min. duration of the Participant's code comprehension as the time window
        time_window = 4                                                                                                                                                                                                                                                    

        # calculate EEG Signal duration
        eeg_duration = raw.n_times / raw.info['sfreq']

        # calculate midpoint of the signal
        midpoint = eeg_duration/2

        #Set time window's starting and end point to choose the middle part of the EEG signal
        tmin = midpoint - (time_window/2)
        tmax = midpoint + (time_window/2)
        
        raw_epochs = mne.Epochs(tmp_raw, events, event_id = 1, tmin=tmin, tmax=tmax, baseline=None, preload=True)
        # Append the epochs to the list
        subject_epochs.append(raw_epochs)

    # Store the epochs for this subject
    epochs[algorithm] = subject_epochs



  0%|          | 0/32 [00:00<?, ?it/s]

Processing  IsPrime
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant01/IsPrimecode_eeg_raw.fif...
    Range : 6629 ... 12819 =     13.258 ...    25.638 secs
Ready.
Reading 0 ... 6190  =      0.000 ...    12.380 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 825 samples (1.650 s)

Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 2000 original time poi

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 2001 original time points ...
0 bad epochs dropped
Processing  IsAnagram
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant01/IsAnagramcode_eeg_raw.fif...
    Range : 133679 ... 188469 =    267.358 ...   376.938 secs
Ready.
Reading 0 ... 54790  =      0.000 ...   109.580 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- F

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


...

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 2001 original time points ...
0 bad epochs dropped
Processing  ReverseQueue
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant01/ReverseQueuecode_eeg_raw.fif...
    Range : 1880709 ... 1889599 =   3761.418 ...  3779.198 secs
Ready.
Reading 0 ... 8890  =      0.000 ...    17.780 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)


- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 825 samples (1.650 s)

Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 2001 original time points ...
0 bad epochs dropped
Processing  Rectangle
Opening raw data file C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant01/Rectanglecode_eeg_raw.fif...
    Range : 1981149 ... 1989679 =   3962.298 ...  3979.358 secs
Ready.
Reading 0 ... 8530  =      0.000 ...    17.060 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, ze

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43920\662105806.py:14: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


In [33]:
df_epoch = pd.DataFrame(columns=['Algorithm', 'Epoch'])
for alg, alg_epochs in epochs.items():
    for i, raw_epochs in enumerate(alg_epochs):
        if raw_epochs is not None:  # Check if epochs are available for this raw file
            epoch = raw_epochs.get_data()
            algorithm = alg
            df_epoch.loc[len(df_epoch)] = [algorithm, epoch]

In [43]:
df_epoch_old = pd.read_csv(result_path+'/epochs.csv')
df_epoch_old.drop('Unnamed: 0', axis = 1)

,Participant,SkillLevel,Epoch
0,1,Intermediate,[[[-5.78384684e-07 -4.33310605e-07 -1.86929033...
1,1,Intermediate,[[[ 1.96516560e-07 2.40960712e-08 -1.69045139...
2,1,Intermediate,[[[ 5.09211951e-07 3.61983725e-07 2.11866564...
3,1,Intermediate,[[[-5.05481545e-08 9.29485285e-08 2.45610570...
4,1,Intermediate,[[[-1.30959641e-07 -4.29617859e-08 4.98843862...
...,...,...,...
1067,71,Expert,[[[-5.61349850e-07 -3.62137125e-07 -7.50861391...
1068,71,Expert,[[[ 2.42115448e-07 2.45864407e-07 3.02057795...
1069,71,Expert,[[[-1.70661097e-08 -8.46787924e-08 -9.76528316...
1070,71,Expert,[[[ 1.07101263e-06 1.08773288e-06 1.03779081...


In [50]:
df_raw_epoch= df_epoch_old[df_epoch_old['Participant'] == 1]. drop('Unnamed: 0', axis = 1)
df_raw_epoch.shape

(32, 3)

In [52]:
df_epoch = df_raw_epoch.copy()

In [54]:
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

In [55]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import make_pipeline
clf = LDA()
pipeline = make_pipeline(csp,clf)


In [27]:

X= epochs.get_data()
y= epochs.events[:, -1]
X.shape, y.shape

AttributeError: 'dict' object has no attribute 'get_data'

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train,  y_test = train_test_split( X, y, test_size= 0.2, random_state= 42)
pipeline.fit(X_train, y_train)

ValueError: With n_samples=1, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [24]:
X= epochs.get_data()

AttributeError: 'dict' object has no attribute 'get_data'

In [13]:
fwd= mne.make_forward_solution(raw.info, trans= 'fsaverage-trans.fif', src='fsaverage-src.fif',
                               bem= 'fsaverage-bem-sol.fif')


OSError: trans file "fsaverage-trans.fif" not found